# Hierarchical circuits
> Let's discuss hierarchical circuits

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from kfnetlist import HierarchicalNetlist, Netlist, NetlistPort, PortRef

import sax

## Models
create a dictionary of models to be used

In [ ]:
models = {
    "coupler": sax.models.coupler_ideal,
    "waveguide": sax.models.straight,
}

## Flat Circuit

Probably best to start from a reference circuit. Let's build a flat MZI netlist (introduced in the SAX Quick Start):

In [ ]:
netlist = Netlist()
for name, component in (("lft", "coupler"), ("top", "waveguide"),
                        ("btm", "waveguide"), ("rgt", "coupler")):
    netlist.create_inst(name, "", component)
for a, b in (("lft,out0", "btm,in0"), ("btm,out0", "rgt,in0"),
             ("lft,out1", "top,in0"), ("top,out0", "rgt,in1")):
    netlist.create_net(PortRef(*a.split(",")), PortRef(*b.split(",")))
for name, endpoint in (("in0", "lft,in0"), ("in1", "lft,in1"),
                       ("out0", "rgt,out0"), ("out1", "rgt,out1")):
    netlist.create_port(name)
    netlist.create_net(NetlistPort(name), PortRef(*endpoint.split(",")))

we can easily simulate this netlist as we did before:

In [ ]:
# created the circuit function
mzi, _ = sax.circuit(netlist=netlist, models=models)

# simulate
wl = jnp.linspace(1.5, 1.6)
result = mzi(wl=wl, top={"length": 20})

# plot
plt.plot(wl, abs(result["in0", "out0"]) ** 2)
plt.xlabel("Wavelength [um]")
plt.ylabel("T")
plt.show()

## Hierarchical Circuit

We can quite easily convert this into a hierarchical netlist:

In [ ]:
top_level = Netlist()
for name in ("top_lft", "btm_rgt"):
    top_level.create_inst(name, "", name, netlist_id=name)
for a, b in (("top_lft,out0", "btm_rgt,in0"),
             ("top_lft,out1", "btm_rgt,in1")):
    top_level.create_net(PortRef(*a.split(",")), PortRef(*b.split(",")))
for name, endpoint in (("in0", "top_lft,in0"), ("in1", "top_lft,in1"),
                       ("out0", "btm_rgt,out0"), ("out1", "btm_rgt,out1")):
    top_level.create_port(name)
    top_level.create_net(NetlistPort(name), PortRef(*endpoint.split(",")))

top_lft = Netlist()
top_lft.create_inst("lft", "", "coupler")
top_lft.create_inst("top", "", "waveguide")
top_lft.create_net(PortRef("lft", "out1"), PortRef("top", "in0"))
for name, endpoint in (("in0", "lft,in0"), ("in1", "lft,in1"),
                       ("out0", "lft,out0"), ("out1", "top,out0")):
    top_lft.create_port(name)
    top_lft.create_net(NetlistPort(name), PortRef(*endpoint.split(",")))

btm_rgt = Netlist()
btm_rgt.create_inst("btm", "", "waveguide")
btm_rgt.create_inst("rgt", "", "coupler")
btm_rgt.create_net(PortRef("btm", "out0"), PortRef("rgt", "in0"))
for name, endpoint in (("in0", "btm,in0"), ("in1", "rgt,in1"),
                       ("out0", "rgt,out0"), ("out1", "rgt,out1")):
    btm_rgt.create_port(name)
    btm_rgt.create_net(NetlistPort(name), PortRef(*endpoint.split(",")))

hierarchical_netlist = HierarchicalNetlist({
    "top_level": top_level, "top_lft": top_lft, "btm_rgt": btm_rgt,
})

and simulate it just as before

In [ ]:
# created the circuit function
mzi, _ = sax.circuit(netlist=hierarchical_netlist, models=models)

# simulate
wl = jnp.linspace(1.5, 1.6)
result = mzi(wl=wl, top_lft={"top": {"length": 20}})

# plot
plt.plot(wl, abs(result["in0", "out0"]) ** 2)
plt.xlabel("Wavelength [um]")
plt.ylabel("T")
plt.show()